# Tu primer modelo de producción minera en Python
### Planeamiento minero — Python aplicado a minería · Nilson R. Garrido

Cada mes, una operación cierra su **reporte de producción**: cuánto metal se recuperó y cuánto vale. Detrás de ese reporte hay una sola identidad, la **contabilidad metalúrgica**, que en Python son unas veinte líneas. Pero un número puntual esconde las decisiones. En este taller construimos ese modelo para un pórfido **Cu-Au** y lo endurecemos hasta que sirva para decidir: **ley de corte**, **análisis de sensibilidad** y **cuantificación de la incertidumbre** con Monte Carlo.

> **Stack:** Python · NumPy · pandas · Matplotlib

## 1) El reporte de producción y la identidad que lo sostiene

El planeamiento de corto plazo se mide contra una pregunta simple de enunciar y difícil de cumplir: **¿cuánto metal pagable produjo la operación este mes, y cuánto vale?** La respuesta no sale de una planta piloto ni de un modelo de caja negra: sale de una identidad contable que se calcula igual en toda mina, la **contabilidad metalúrgica**.

El aporte de Python no es reemplazar esa identidad, sino **convertirla en un modelo**: parametrizado, reproducible y, sobre todo, interrogable. Un número puntual (5,879 t de Cu) parece una certeza; el modelo permite preguntarle qué lo mueve, dónde está el punto de equilibrio y con qué probabilidad se cumple. Ese salto, de un cálculo a una herramienta de decisión, es el objetivo del taller.

## 2) Marco teórico

### 2.1) La identidad de contabilidad metalúrgica

El metal recuperado en un periodo es la suma, sobre cada lote de mineral procesado, del producto de tres factores: cuánto material se trató, qué tan rico era y qué fracción se logró recuperar en planta:

$$ \text{Metal recuperado} = \sum_i T_i \cdot g_i \cdot R_i $$

donde $T_i$ es el tonelaje del lote (t), $g_i$ la ley (fracción de metal) y $R_i$ la recuperación metalúrgica (fracción). El detalle que hace fallar a la mitad de las hojas de cálculo es el **manejo de unidades**: la ley de Cu se reporta en **%** (hay que dividir entre 100) y la de Au en **g/t** (hay que pasar de gramos a onzas troy, 31.1035 g/oz). El metal de Cu se expresa en toneladas o libras (2204.62 lb/t) y el de Au en onzas.

### 2.2) De metal a valor: pagables y precio

El metal recuperado en concentrado no se cobra completo. La fundición paga una **fracción pagable** (típicamente 96 % del Cu, 95 % del Au) y descuenta cargos de tratamiento y refinación (TC/RC). El ingreso aproximado por metal es:

$$ \text{Ingreso} = \text{Metal}_{Cu}\cdot P_{Cu}\cdot f_{Cu} + \text{Metal}_{Au}\cdot P_{Au}\cdot f_{Au} $$

con $P$ el precio y $f$ el factor pagable. Es un **NSR** (Net Smelter Return) simplificado, suficiente para dimensionar el valor de la producción y el peso de cada metal.

### 2.3) Ley de corte (cut-off)

La **ley de corte** es la ley mínima a la que un lote paga su propio costo de tratamiento. Igualando ingreso y costo por tonelada se despeja:

$$ g_{corte} = \frac{\text{Costo}}{P_{Cu}\cdot R_{Cu}\cdot f_{Cu}} $$

El **crédito por subproducto** (el Au) baja la ley de corte del Cu: cada tonelada trae onzas de oro que ayudan a pagar el costo, así que se acepta como mineral material de menor ley de cobre.

### 2.4) Sensibilidad e incertidumbre

El modelo determinístico entrega un número, pero las variables de entrada (ley, recuperación, tonelaje, precio) son estimaciones con rango. Dos preguntas de planeamiento nacen de ahí:

- **Sensibilidad:** ¿qué variable mueve más el resultado? Se responde con un **diagrama de tornado**, perturbando cada entrada dentro de su rango plausible. La intuición de campo (mover más tonelaje) suele perder frente a la ley y el precio.
- **Incertidumbre:** ¿con qué probabilidad se cumple la meta? Se responde con **simulación de Monte Carlo**: se muestrean las entradas según su distribución y se obtiene la distribución del metal producido, con sus percentiles P10, P50 y P90.

## 3) Datos: un mes de producción Cu-Au

### 3.1) Variables

Generamos un mes de operación de un pórfido Cu-Au, con un registro por día:

| Variable | Símbolo | Unidad | Descripción |
|---|---|---|---|
| Tonelaje | T | t/día | Mineral tratado en planta |
| Ley de cobre | g_Cu | % | Ley de cabeza de Cu |
| Ley de oro | g_Au | g/t | Ley de cabeza de Au |
| Recuperación Cu | R_Cu | fracción | Recuperación metalúrgica de Cu |
| Recuperación Au | R_Au | fracción | Recuperación metalúrgica de Au |

### 3.2) Parámetros comerciales y de costo

| Parámetro | Valor | Nota |
|---|---|---|
| Precio Cu | 4.00 USD/lb | Precio de referencia |
| Precio Au | 2400 USD/oz | Precio de referencia |
| Pagable Cu / Au | 0.96 / 0.95 | Fracción que paga la fundición |
| Costo | 18 USD/t | Mina + planta + G&A por tonelada tratada |

La ley y la recuperación **co-varían**: un día con mineral más rico suele recuperar algo mejor. El oro acompaña al cobre, como en todo pórfido Cu-Au.

## 4) Implementación en Python

### 4.1) Librerías y parámetros

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parametros comerciales y de costo
PRECIO_CU, PRECIO_AU = 4.00, 2400.0    # USD/lb Cu, USD/oz Au
PAG_CU, PAG_AU       = 0.96, 0.95      # fraccion pagable
COSTO                = 18.0            # USD/t tratada (mina + planta + G&A)
LB_POR_T, G_POR_OZ   = 2204.62, 31.1035

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)

### 4.2) Generación del mes de producción

Un factor latente de calidad del mineral del día correlaciona la ley de Cu, la de Au y las recuperaciones. Guardamos el dataset para el repositorio.

In [ ]:
N = 30   # dias del mes
f = rng.normal(size=N)                                                  # calidad latente del dia
tonelaje = np.clip(rng.normal(40000, 3200, N), 30000, 50000)            # t/dia
cu = np.clip(np.exp(np.log(0.55) + 0.18 * (0.7*f + 0.7*rng.normal(size=N))), 0.2, 1.2)   # %
au = np.clip(np.exp(np.log(0.18) + 0.22 * (0.65*f + 0.75*rng.normal(size=N))), 0.03, 0.8) # g/t
rec_cu = np.clip(0.865 + 0.03*(cu-cu.mean())/cu.std() + 0.010*rng.normal(size=N), 0.80, 0.92)
rec_au = np.clip(0.700 + 0.03*(au-au.mean())/au.std() + 0.015*rng.normal(size=N), 0.60, 0.80)

prod = pd.DataFrame({'dia': np.arange(1, N+1),
                     'tonelaje_t': np.round(tonelaje).astype(int),
                     'cu_pct': np.round(cu, 3), 'au_gpt': np.round(au, 3),
                     'rec_cu': np.round(rec_cu, 3), 'rec_au': np.round(rec_au, 3)})

import os
os.makedirs('../data/raw', exist_ok=True)
prod.to_csv('../data/raw/produccion_mes.csv', index=False)
prod.head(8)

### 4.3) El modelo, en veinte líneas

Este es el núcleo. Con el dataset cargado, la contabilidad metalúrgica del mes cabe en un bloque: metal recuperado por día, totales del mes e ingreso. Todo lo demás en el taller se construye sobre esto.

In [ ]:
df = pd.read_csv('../data/raw/produccion_mes.csv')

# metal recuperado por dia (cuidado con las unidades: Cu en %, Au en g/t)
df['cu_t']  = df.tonelaje_t * df.cu_pct/100      * df.rec_cu             # t de Cu
df['au_oz'] = df.tonelaje_t * df.au_gpt/G_POR_OZ * df.rec_au             # oz de Au

# totales del mes
cu_t   = df.cu_t.sum()
au_oz  = df.au_oz.sum()
cu_lb  = cu_t * LB_POR_T

# valor (NSR simplificado)
ing_cu  = cu_lb * PRECIO_CU * PAG_CU
ing_au  = au_oz * PRECIO_AU * PAG_AU
ingreso = ing_cu + ing_au

print(f'Cu recuperado:  {cu_t:8,.0f} t   ({cu_lb/1e6:5.2f} Mlb)')
print(f'Au recuperado:  {au_oz:8,.0f} oz')
print(f'Ingreso:        US$ {ingreso/1e6:6.1f} M   (Cu {ing_cu/1e6:.1f} + Au {ing_au/1e6:.1f})')

Ahí está el modelo de producción funcionando: **5,879 t de Cu** (12.96 Mlb), **5,106 oz de Au** y un ingreso de **US$ 61.4 M**. Un solo bloque de código reproduce el número que ocupa la primera línea del reporte mensual. A partir de aquí lo interrogamos.

## 5) La producción del mes

### 5.1) Cómo se construye el tonelaje de metal día a día

El total del mes es la acumulación de la producción diaria. Ver el aporte de cada día muestra la variabilidad real de una operación (días ricos y días pobres) que el promedio esconde.

In [ ]:
ley_media_cu = (df.tonelaje_t * df.cu_pct).sum() / df.tonelaje_t.sum()
ley_media_au = (df.tonelaje_t * df.au_gpt).sum() / df.tonelaje_t.sum()
print(f'Tonelaje del mes: {df.tonelaje_t.sum():,} t')
print(f'Ley media ponderada  Cu: {ley_media_cu:.3f} %   Au: {ley_media_au:.3f} g/t')

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(df.dia, df.cu_t, color='#0f766e', alpha=0.85, label='Cu por día (t)')
ax2 = ax.twinx()
ax2.plot(df.dia, df.cu_t.cumsum(), color='#b4312a', lw=2.5, marker='o', ms=3, label='Cu acumulado (t)')
ax.set_xlabel('Día del mes'); ax.set_ylabel('Cu del día (t)', color='#0f766e')
ax2.set_ylabel('Cu acumulado del mes (t)', color='#b4312a')
ax.set_title('Producción de Cu: aporte diario y acumulado')
plt.tight_layout(); plt.show()

### 5.2) De dónde viene el valor

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
barras = ax.barh(['Cobre', 'Oro'], [ing_cu/1e6, ing_au/1e6], color=['#0f766e', '#c79a3a'])
for b, v in zip(barras, [ing_cu/1e6, ing_au/1e6]):
    ax.text(v + 0.6, b.get_y()+b.get_height()/2, f'US$ {v:.1f} M ({v/(ingreso/1e6):.0%})',
            va='center', fontsize=11, fontweight='bold', color='#16241d')
ax.set_xlabel('Ingreso (US$ M)'); ax.set_xlim(0, ing_cu/1e6*1.25)
ax.set_title(f'Ingreso del mes: US$ {ingreso/1e6:.1f} M'); ax.invert_yaxis()
for s in ['top', 'right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

El cobre aporta el **81 %** del ingreso y el oro el **19 %**. El oro es subproducto, pero un subproducto que no se desprecia: paga por sí solo una parte relevante del costo, y eso reaparece en la ley de corte.

## 6) Ley de corte: qué es mineral

La ley de corte es el umbral que separa mineral de estéril. La calculamos sin y con el crédito del oro, para ver cuánto baja el subproducto la exigencia sobre el cobre.

In [ ]:
precio_cu_t = PRECIO_CU * LB_POR_T / 100    # USD por (t . 1% Cu) recuperado y pagable

# credito de Au por tonelada (con la ley y recuperacion medias)
credito_au = ley_media_au / G_POR_OZ * df.rec_au.mean() * PRECIO_AU * PAG_AU   # USD/t

cutoff_sin = COSTO / (precio_cu_t * df.rec_cu.mean() * PAG_CU)
cutoff_con = (COSTO - credito_au) / (precio_cu_t * df.rec_cu.mean() * PAG_CU)

print(f'Crédito por Au:            US$ {credito_au:.2f} / t')
print(f'Ley de corte Cu sin Au:    {cutoff_sin:.3f} %')
print(f'Ley de corte Cu con Au:    {cutoff_con:.3f} %')
print(f'Ley de cabeza media Cu:    {ley_media_cu:.3f} %  ->  {ley_media_cu/cutoff_con:.1f}x la ley de corte')

El crédito del oro (**US$ 9.3/t**) baja la ley de corte del cobre de **0.246 %** a **0.119 %**. La ley de cabeza (0.543 % Cu) corre a más de **4 veces** la ley de corte con crédito: el mes opera con margen holgado. Ese múltiplo es el primer indicador de salud económica de la alimentación.

## 7) Análisis de sensibilidad: ¿qué mueve el resultado?

Perturbamos cada variable dentro de un rango plausible y medimos el efecto sobre el ingreso. El resultado se ordena en un **diagrama de tornado**: la barra más larga es la variable que más manda.

In [ ]:
def ingreso_mes(fcu=1, fau=1, frcu=1, fton=1, fpcu=1, fpau=1):
    ct = (df.tonelaje_t*fton * df.cu_pct*fcu/100      * np.clip(df.rec_cu*frcu, 0, 1)).sum()
    ao = (df.tonelaje_t*fton * df.au_gpt*fau/G_POR_OZ * df.rec_au).sum()
    return ct*LB_POR_T*PRECIO_CU*fpcu*PAG_CU + ao*PRECIO_AU*fpau*PAG_AU

drivers = [('Precio Cu', 0.20, 'fpcu'), ('Ley Cu', 0.15, 'fcu'), ('Tonelaje', 0.08, 'fton'),
           ('Recuperación Cu', 0.04, 'frcu'), ('Ley Au', 0.15, 'fau'), ('Precio Au', 0.15, 'fpau')]
base = ingreso_mes()
filas = []
for nombre, r, k in drivers:
    hi = ingreso_mes(**{k: 1+r}); lo = ingreso_mes(**{k: 1-r})
    filas.append((nombre, r, (lo-base)/1e6, (hi-base)/1e6, (hi-lo)/1e6))
tor = pd.DataFrame(filas, columns=['driver','rango','baja_M','sube_M','swing_M']).sort_values('swing_M')

fig, ax = plt.subplots(figsize=(10, 5))
for _, r in tor.iterrows():
    ax.barh(r.driver, r.sube_M, color='#0f766e'); ax.barh(r.driver, r.baja_M, color='#b4312a')
    ax.text(r.sube_M+0.2, r.driver, f'+{r.sube_M:.1f}', va='center', fontsize=9, color='#0f766e')
    ax.text(r.baja_M-0.2, r.driver, f'{r.baja_M:.1f}', va='center', ha='right', fontsize=9, color='#b4312a')
ax.axvline(0, color='#16241d', lw=1)
ax.set_xlabel('Cambio en el ingreso del mes (US$ M)')
ax.set_title(f'Sensibilidad del ingreso (base US$ {base/1e6:.1f} M)')
for s in ['top','right','left']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
print(tor[['driver','rango','swing_M']].to_string(index=False))

El orden es revelador: **el precio del Cu y la ley de Cu dominan**, muy por encima del tonelaje. La reacción operativa instintiva ante una meta de metal es *mover más material*, pero el tonelaje (±8 %) pesa menos que la ley (±15 %) y bastante menos que el precio (±20 %). Donde más se gana es asegurando la **ley alimentada** (control de leyes, dilución) y gestionando la **exposición al precio**, no acelerando la pala.

## 8) Incertidumbre: ¿con qué probabilidad se cumple la meta?

El modelo determinístico da 5,879 t de Cu como si fuera un hecho. Pero la ley proviene de un modelo de recursos con error, la recuperación varía y el tonelaje depende de la disponibilidad de planta. Propagamos esa **incertidumbre sistemática** (a nivel de mes, no ruido diario) con Monte Carlo.

In [ ]:
cu_t_base = df.cu_t.sum()
sim_rng = np.random.default_rng(7)
M = 50_000
f_ley = sim_rng.lognormal(0, 0.09, M)     # incertidumbre del modelo de leyes (~9%)
f_rec = sim_rng.normal(1, 0.025, M)       # recuperación de planta (~2.5%)
f_ton = sim_rng.normal(1, 0.04, M)        # throughput (~4%)
cu_sim = cu_t_base * f_ley * f_rec * f_ton

p10, p50, p90 = np.percentile(cu_sim, [10, 50, 90])
META = 5400   # compromiso al que se quiere comprometer planeamiento
print(f'Cu mensual  P10 {p10:,.0f}  |  P50 {p50:,.0f}  |  P90 {p90:,.0f} t')
print(f'Prob de cumplir el PLAN determinístico ({cu_t_base:,.0f} t): {(cu_sim>=cu_t_base).mean():.0%}')
print(f'Prob de cumplir un compromiso de {META:,} t:               {(cu_sim>=META).mean():.0%}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.hist(cu_sim, bins=60, color='#93c9b7', edgecolor='white', alpha=0.9)
for v, c, lab in [(p10,'#b4312a','P10'), (p50,'#16241d','P50'), (p90,'#0f766e','P90')]:
    ax.axvline(v, color=c, lw=2, ls='--'); ax.text(v, ax.get_ylim()[1]*0.96, f'{lab}\n{v:,.0f}',
            ha='center', va='top', fontsize=9, fontweight='bold', color=c)
ax.axvline(cu_t_base, color='#c79a3a', lw=2.5); ax.text(cu_t_base, ax.get_ylim()[1]*0.55,
        f' Plan\n {cu_t_base:,.0f} t', fontsize=9, fontweight='bold', color='#8a6d1f')
ax.set_xlabel('Cu producido en el mes (t)'); ax.set_ylabel('Frecuencia (simulaciones)')
ax.set_title('Distribución de la producción mensual de Cu (Monte Carlo, 50,000 escenarios)')
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

Este es el resultado que cambia la conversación de planeamiento: el plan determinístico (5,879 t) tiene apenas **50 % de probabilidad de cumplirse**, porque es el centro de la distribución. Comprometer ese número es, literalmente, lanzar una moneda. Un compromiso defendible se toma **más abajo**: 5,400 t se cumplen con **~80 %** de probabilidad (P80), el estándar habitual para comprometer producción. La diferencia entre el P50 y el P80 es la **reserva de riesgo** que el número puntual no muestra.

## 9) Reconciliación: el modelo contra la realidad

El cierre del ciclo es la **reconciliación**: comparar lo planeado con lo realmente producido mediante **factores** (mina/modelo y planta/mina). Un factor de reconciliación cercano a 1.0 valida el modelo; una desviación sistemática lo corrige. Con un mes real hipotético:

In [ ]:
# 'real' del mes (lo que reporto planta): tonelaje y metal medidos
cu_real_t = 5610.0     # t de Cu segun balance de planta
factor_metal = cu_real_t / cu_t_base
print(f'Cu plan (modelo):  {cu_t_base:,.0f} t')
print(f'Cu real (planta):  {cu_real_t:,.0f} t')
print(f'Factor de reconciliación (real/plan): {factor_metal:.3f}')
estado = 'dentro de tolerancia (+/- 5%)' if abs(factor_metal-1) <= 0.05 else 'fuera de tolerancia: revisar modelo'
print(f'Estado: {estado}')

Un factor de **0.954** cae dentro de la tolerancia habitual de ±5 %: el modelo predice bien y no requiere corrección. Fuera de ese rango, la desviación sistemática apunta a dilución no contabilizada, un modelo de recursos sesgado o una recuperación distinta a la supuesta. La reconciliación es lo que mantiene honesto al modelo mes a mes.

## 10) Conclusiones

- La **contabilidad metalúrgica** (tonelaje × ley × recuperación, con las unidades bien puestas) es un modelo de producción completo en unas veinte líneas: **5,879 t de Cu, 5,106 oz de Au, US$ 61.4 M**.
- El **oro subproducto** aporta el 19 % del ingreso y baja la ley de corte del cobre de 0.246 % a 0.119 %: un subproducto cambia qué material es mineral.
- La **sensibilidad** ordena las palancas: precio y ley de Cu mandan; el tonelaje pesa menos que la intuición de campo. El foco de valor está en la ley alimentada y la exposición al precio.
- La **incertidumbre** es la lección central: el plan determinístico se cumple solo el 50 % de las veces. Comprometer producción exige bajar al P80 (5,400 t), donde la probabilidad es defendible.
- La **reconciliación** cierra el ciclo y mantiene calibrado el modelo.

El modelo de veinte líneas no es el final: es el esqueleto sobre el que se montan la ley de corte, la sensibilidad, la incertidumbre y la reconciliación. Ahí está la diferencia entre un cálculo y una herramienta de planeamiento.

## 11) Referencias

- Hustrulid, W., Kuchta, M., & Martin, R. (2013). *Open Pit Mine Planning and Design* (3rd ed.). CRC Press.
- Wills, B. A., & Finch, J. A. (2016). *Wills' Mineral Processing Technology* (8th ed.). Butterworth-Heinemann.
- Rendu, J.-M. (2014). *An Introduction to Cut-off Grade Estimation* (2nd ed.). SME.
- Lane, K. F. (1988). *The Economic Definition of Ore: Cut-off Grades in Theory and Practice*. Mining Journal Books.
- Morley, C. (2003). Beyond reconciliation: a proactive approach to using mining data. *Mining Magazine*.

---
*Nilson Rolando Garrido Asenjo · Ingeniero de Minas · Python para Minería · [linkedin.com/in/nrgarridoa](https://www.linkedin.com/in/nrgarridoa) · [nrgarridoa.github.io](https://nrgarridoa.github.io)*